### DUPLICATE RECORD CHECK

In [1]:
import pandas as pd
import numpy as np
df = pd.read_excel('data/EDA_Practice_Customer_Orders_150.xlsx', skiprows=3)

df.columns = df.columns.str.strip()

str_cols = df.select_dtypes(include=["object"]).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace("nan", np.nan)

n_full = df.duplicated().sum()
dup_rows = df[df.duplicated(keep=False)].sort_values("Order_ID")

In [2]:
checks = {
    "Full-Row Duplicates": n_full,
    "Order_ID Duplicates (Exact + Conflicts)": df["Order_ID"].duplicated().sum(),
}

clean = df.copy()
clean = clean.drop_duplicates()
clean = clean.drop_duplicates(subset=["Order_ID"], keep="first")

markers = ["Missing", "N/A", "Unknown", ""]
clean = clean.replace(markers, np.nan)


In [3]:
for col in ["Unit_Price", "Total_Amount", "Discount_Pct"]:
    if col in clean.columns:
        clean[col] = clean[col].astype(str).str.replace(",", "", regex=False)
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

numeric_cols = ["Customer_Age", "Quantity", "Customer_Rating", "Delivery_Days"]
for col in numeric_cols:
    if col in clean.columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

if "Order_Date" in clean.columns:
    clean["Order_Date"] = pd.to_datetime(clean["Order_Date"], errors="coerce")

In [16]:
mask = (
    clean["Total_Amount"].isna()
    & clean["Quantity"].notna()
    & clean["Unit_Price"].notna()
    & clean["Discount_Pct"].notna()
)

clean.loc[mask, "Total_Amount"] = (
    clean.loc[mask, "Quantity"]
    * clean.loc[mask, "Unit_Price"]
    * (1 - clean.loc[mask, "Discount_Pct"] / 100)
).round(2)

print(f"Original rows loaded: {len(df)}")
print(f"Full-row duplicates found: {n_full}")
print(f"Final cleaned rows: {len(clean)}")

Original rows loaded: 150
Full-row duplicates found: 3
Final cleaned rows: 147


In [6]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
with pd.ExcelWriter(
    "deliverables/Day-13_Duplicate_and_Cleaning_Report.xlsx", engine="openpyxl"
) as writer:
    pd.DataFrame(
        {"Check": list(checks.keys()), "Count": list(checks.values())}
    ).to_excel(writer, "Duplicate_Summary", index=False)

    dup_rows.to_excel(writer, "Flagged_Duplicates", index=False)

clean.to_csv("deliverables/Customer_Orders_Cleaned.csv", index=False)

print("\n✅ Success! Files generated:")
print("1. Day-13_Duplicate_and_Cleaning_Report.xlsx")
print("2. Customer_Orders_Cleaned.csv")


✅ Success! Files generated:
1. Day-13_Duplicate_and_Cleaning_Report.xlsx
2. Customer_Orders_Cleaned.csv
